# Preliminary Paper Runbook (Colab)

This notebook is the resumable Colab entrypoint for fast preliminary paper iteration.

What it does:
1. Mounts Google Drive and resolves the repo root.
2. Creates a run-specific runtime config under `runs/experiments/<run_id>/configs/`.
3. Uses a stage manifest so completed expensive stages are skipped on rerun.
4. Tracks ETA per stage and shows remaining ETA before and after execution.
5. Defaults to a lean benchmark + training path, with heavier paper stages available as opt-in.

Profiles:
- `RUN_PROFILE = "fast"` keeps the default path practical for Colab reruns.
- `RUN_PROFILE = "full"` restores the broader paper pipeline.

Resume behavior:
- If a stage already completed and its outputs still exist, rerunning the pipeline skips it.
- If a stage fails, fix the issue and rerun the notebook or just rerun the pipeline cell.
- You do not need to rerun completed expensive stages.


In [1]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = None
candidates = [
    Path(os.environ.get("FRM_REPO_DIR", "")).expanduser() if os.environ.get("FRM_REPO_DIR") else None,
    Path.cwd(),
    Path("/content/Forward-Risk-Manager"),
    Path("/content/drive/MyDrive/Forward-Risk-Manager"),
    Path("/content/drive/MyDrive/forward-risk-manager"),
]
for candidate in candidates:
    if candidate is None:
        continue
    if (candidate / "configs" / "default.toml").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError("Could not locate repo root containing configs/default.toml")

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from frisk.notebook_runtime import (
    NotebookStage,
    format_duration,
    remaining_eta_seconds,
    run_tracked_command,
    shell_quote,
    stage_status_rows,
    step_is_complete,
    write_toml_overrides,
)

print("repo root:", ROOT)
print("cwd:", Path.cwd())


Mounted at /content/drive
repo root: /content/drive/MyDrive/Forward-Risk-Manager
cwd: /content/drive/MyDrive/Forward-Risk-Manager


In [2]:
from datetime import datetime, timezone
import importlib.util
import json

import pandas as pd

PYTHON_EXE = shell_quote(sys.executable)

RUN_PROFILE = "fast"  # "fast" | "full"
RUN_ID_OVERRIDE = ""
RESUME_POLICY = "auto"  # "auto" | "force_new" | "force_resume"
DEVICE = "cuda"
INSTALL_DEPS = IN_COLAB

if RUN_PROFILE not in {"fast", "full"}:
    raise ValueError(f"Unsupported RUN_PROFILE: {RUN_PROFILE}")

# Profile-populated defaults. You can still edit any toggle below after choosing RUN_PROFILE.
PROFILE_STAGE_DEFAULTS = {
    "fast": {
        "RUN_BUILD_GRAPHS": False,
        "RUN_BENCHMARK": True,
        "RUN_TRAIN": True,
        "RUN_SWEEP": False,
        "RUN_DUAL_SCORE": False,
        "RUN_SCENARIO": False,
        "RUN_BACKTEST": False,
        "RUN_PUBLISH": False,
    },
    "full": {
        "RUN_BUILD_GRAPHS": False,
        "RUN_BENCHMARK": True,
        "RUN_TRAIN": True,
        "RUN_SWEEP": True,
        "RUN_DUAL_SCORE": True,
        "RUN_SCENARIO": True,
        "RUN_BACKTEST": True,
        "RUN_PUBLISH": False,
    },
}
STAGE_DEFAULTS = PROFILE_STAGE_DEFAULTS[RUN_PROFILE].copy()
RUN_BUILD_GRAPHS = STAGE_DEFAULTS["RUN_BUILD_GRAPHS"]
RUN_BENCHMARK = STAGE_DEFAULTS["RUN_BENCHMARK"]
RUN_TRAIN = STAGE_DEFAULTS["RUN_TRAIN"]
RUN_SWEEP = STAGE_DEFAULTS["RUN_SWEEP"]
RUN_DUAL_SCORE = STAGE_DEFAULTS["RUN_DUAL_SCORE"]
RUN_SCENARIO = STAGE_DEFAULTS["RUN_SCENARIO"]
RUN_BACKTEST = STAGE_DEFAULTS["RUN_BACKTEST"]
RUN_PUBLISH = STAGE_DEFAULTS["RUN_PUBLISH"]

BASE_CONFIG = ROOT / "configs" / "paper_final_500.toml"
DEFAULT_CONFIG = ROOT / "configs" / "default.toml"
PREBUILT_GRAPH_PATH = ROOT / "data" / "processed" / "graphs_master_ff_rich.pt"
SHARDED_GRAPH_PATH = Path(str(PREBUILT_GRAPH_PATH) + ".sharded")
FAST_BENCHMARK_MODES = ["ff_layerwise", "ff_e2e", "backprop_contrastive"]
BENCHMARK_MODES = FAST_BENCHMARK_MODES.copy() if RUN_PROFILE == "fast" else []

FAST_TRAIN_OVERRIDES = {
    "epochs": 60,
    "graph_limit": 0,
    "graph_limit_keep_recent": True,
    "epoch_graph_fraction": 0.25,
    "epoch_graph_min": 1024,
    "epoch_graph_mode": "recent_bias",
    "torch_compile": False,
    "auto_tune_batch": True,
}
FAST_BENCHMARK_OVERRIDES = {
    "epochs": 15,
    "walk_forward_max_folds": 2,
    "eval_neg_modes": ["time_flip"],
    "timing_warmup_epochs": 1,
}
FAST_SWEEP_OVERRIDES = {
    "max_runs": 8,
    "walk_forward_max_folds_cap": 2,
    "eval_neg_modes": ["time_flip"],
}

PRICE_CANDIDATES = [
    ROOT / "data" / "processed" / "prices.csv",
    ROOT / "data" / "consolidated_ff_local" / "prices.csv",
    ROOT / "data" / "processed_long" / "prices.csv",
]
CONSTITUENT_CANDIDATES = [
    ROOT / "data" / "processed" / "constituents.csv",
    ROOT / "data" / "processed_long" / "constituents.csv",
]
MACRO_CANDIDATES = [
    ROOT / "data" / "processed" / "macro.csv",
    ROOT / "data" / "consolidated_ff_local" / "macro.csv",
]


def first_existing(paths, *, required=True):
    for path in paths:
        if path.exists():
            return path.resolve()
    if required:
        raise FileNotFoundError("No existing path found in: " + ", ".join(str(p) for p in paths))
    return None


def resolve_graph_source_path(base_path: Path) -> tuple[Path, str]:
    sharded_path = Path(str(base_path) + ".sharded")
    manifest_path = sharded_path / "manifest.json"
    if sharded_path.is_dir() and manifest_path.exists():
        return sharded_path.resolve(), "sharded"
    if base_path.exists():
        return base_path.resolve(), "packed"
    raise FileNotFoundError(
        f"Missing graph artifact. Checked packed={base_path} and sharded={sharded_path}"
    )


def estimate_walk_forward_folds(
    n_items: int | None,
    *,
    train_frac: float,
    eval_frac: float,
    step_frac: float,
    min_train: int,
    min_eval: int,
    max_folds: int,
) -> int | None:
    if n_items is None or n_items < 2:
        return None
    train_size = min(max(max(int(round(n_items * float(train_frac))), int(min_train)), 1), n_items - 1)
    eval_size = min(max(max(int(round(n_items * float(eval_frac))), int(min_eval)), 1), n_items - train_size)
    if eval_size <= 0:
        return 0
    step_size = max(1, int(round(n_items * float(step_frac)))) if float(step_frac) > 0 else eval_size
    folds = 0
    eval_start = train_size
    while eval_start + eval_size <= n_items:
        folds += 1
        if max_folds > 0 and folds >= int(max_folds):
            break
        eval_start += step_size
    return folds


def load_toml_section(path: Path, section: str) -> dict:
    try:
        import tomllib
    except ModuleNotFoundError:
        import tomli as tomllib
    with path.open("rb") as handle:
        payload = tomllib.load(handle)
    values = payload.get(section, {})
    return dict(values) if isinstance(values, dict) else {}


DATA_PRICES_PATH = first_existing(PRICE_CANDIDATES, required=True)
DATA_CONSTITUENTS_PATH = first_existing(CONSTITUENT_CANDIDATES, required=False)
DATA_MACRO_PATH = first_existing(MACRO_CANDIDATES, required=False)

if RESUME_POLICY == "force_resume" and not RUN_ID_OVERRIDE.strip():
    raise ValueError("RESUME_POLICY='force_resume' requires RUN_ID_OVERRIDE")

resume_requested = bool(RUN_ID_OVERRIDE.strip()) and RESUME_POLICY != "force_new"
if resume_requested:
    RUN_ID = RUN_ID_OVERRIDE.strip()
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID
else:
    RUN_ID = f"paper_prelim_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID

for sub in ("configs", "data", "metrics", "plots", "logs", "models", "diagnostics"):
    (RUN_ROOT / sub).mkdir(parents=True, exist_ok=True)

RUNTIME_CONFIG = RUN_ROOT / "configs" / "runtime_config.toml"
MANIFEST_PATH = RUN_ROOT / "logs" / "stage_manifest.json"
LOG_DIR = RUN_ROOT / "logs" / "stage_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_OUT = RUN_ROOT / "data" / "graphs_preliminary.pt"
GRAPH_RESOLVED_PATH = GRAPH_OUT.resolve() if RUN_BUILD_GRAPHS else resolve_graph_source_path(PREBUILT_GRAPH_PATH)[0]
GRAPH_FORMAT = "packed" if RUN_BUILD_GRAPHS else resolve_graph_source_path(PREBUILT_GRAPH_PATH)[1]
GRAPH_PATH_FOR_RUN = GRAPH_OUT if RUN_BUILD_GRAPHS else PREBUILT_GRAPH_PATH

TRAIN_LOG_CSV = RUN_ROOT / "metrics" / "ff_train.csv"
TRAIN_PLOT = RUN_ROOT / "plots" / "ff_train.png"
MODEL_CKPT = RUN_ROOT / "models" / "ff_model.pt"
ENCODER_CKPT = RUN_ROOT / "models" / "encoder.pt"
CRITIC_CKPT = RUN_ROOT / "models" / "critic.pt"

BENCHMARK_CSV = RUN_ROOT / "metrics" / "benchmark.csv"
BENCHMARK_FOLDS_CSV = RUN_ROOT / "metrics" / "benchmark_walk_forward_folds.csv"
BENCHMARK_BASELINE_CSV = RUN_ROOT / "metrics" / "benchmark_baseline.csv"
BENCHMARK_HISTORY_CSV = RUN_ROOT / "metrics" / "benchmark_history.csv"
BENCHMARK_PLOT = RUN_ROOT / "plots" / "benchmark_speed_sep.png"
BENCHMARK_BAR = RUN_ROOT / "plots" / "benchmark.png"
PAPER_SUMMARY_MD = RUN_ROOT / "logs" / "paper_benchmark_summary.md"
PAPER_SUMMARY_CSV = RUN_ROOT / "metrics" / "paper_benchmark_summary.csv"
PAPER_SUMMARY_JSON = RUN_ROOT / "logs" / "paper_benchmark_summary.json"

SWEEP_CSV = RUN_ROOT / "metrics" / "ff_sweep.csv"
DUAL_SCORE_TXT = RUN_ROOT / "logs" / "dual_score_report.txt"
DUAL_SCORE_CSV = RUN_ROOT / "metrics" / "dual_score_report.csv"

SCENARIO_CSV = RUN_ROOT / "metrics" / "scenario_book.csv"
SCENARIO_DIAG_CSV = RUN_ROOT / "diagnostics" / "scenario_constraint_diagnostics.csv"
STRESS_CSV = RUN_ROOT / "metrics" / "stress_test_report.csv"
STRESS_PLOT = RUN_ROOT / "plots" / "stress_test_report.png"
CALIBRATION_JSON = RUN_ROOT / "diagnostics" / "hallucination_calibration.json"
CALIBRATION_BY_TICKER_CSV = RUN_ROOT / "diagnostics" / "hallucination_calibration_by_ticker.csv"

GOODNESS_CSV = RUN_ROOT / "diagnostics" / "goodness_backtest.csv"
GOODNESS_QUANTILES_CSV = RUN_ROOT / "diagnostics" / "goodness_quantiles.csv"
GOODNESS_PLOT = RUN_ROOT / "plots" / "goodness_scatter.png"
GOODNESS_EVENTS_CSV = RUN_ROOT / "diagnostics" / "goodness_events.csv"
GOODNESS_STRATEGY_CSV = RUN_ROOT / "diagnostics" / "goodness_strategy_metrics.csv"
GOODNESS_TIMELINE_PLOT = RUN_ROOT / "plots" / "goodness_timeline.png"

section_overrides = {
    "build_graphs": {
        "prices": str(DATA_PRICES_PATH),
        "out": str(GRAPH_OUT),
    },
    "train": {
        "graphs": str(GRAPH_PATH_FOR_RUN),
        "device": DEVICE,
        "log_csv": str(TRAIN_LOG_CSV),
        "plot_path": str(TRAIN_PLOT),
        "save_model": str(MODEL_CKPT),
        "save_encoder": str(ENCODER_CKPT),
        "save_critic": str(CRITIC_CKPT),
    },
    "benchmark": {
        "out_csv": str(BENCHMARK_CSV),
        "walk_forward_out_csv": str(BENCHMARK_FOLDS_CSV),
        "baseline_out_csv": str(BENCHMARK_BASELINE_CSV),
        "history_out_csv": str(BENCHMARK_HISTORY_CSV),
        "plot_path": str(BENCHMARK_PLOT),
        "bar_plot_path": str(BENCHMARK_BAR),
        "econ_prices": str(DATA_PRICES_PATH),
    },
    "sweep": load_toml_section(DEFAULT_CONFIG, "sweep"),
    "scenario_book": load_toml_section(DEFAULT_CONFIG, "scenario_book"),
}
section_overrides["sweep"].update({
    "out_csv": str(SWEEP_CSV),
    "econ_prices": str(DATA_PRICES_PATH),
})
section_overrides["scenario_book"].update({
    "critic_model": str(CRITIC_CKPT),
    "diag_out": str(SCENARIO_DIAG_CSV),
    "out": str(SCENARIO_CSV),
})
if DATA_CONSTITUENTS_PATH is not None:
    section_overrides["build_graphs"]["constituents"] = str(DATA_CONSTITUENTS_PATH)
if DATA_MACRO_PATH is not None:
    section_overrides["build_graphs"]["macro"] = str(DATA_MACRO_PATH)

if RUN_PROFILE == "fast":
    section_overrides["train"].update(FAST_TRAIN_OVERRIDES)
    section_overrides["benchmark"].update(FAST_BENCHMARK_OVERRIDES)
    if RUN_SWEEP:
        section_overrides["sweep"].update(FAST_SWEEP_OVERRIDES)

write_toml_overrides(BASE_CONFIG, RUNTIME_CONFIG, section_overrides)

GRAPH_COUNT = None
GRAPH_NUM_SHARDS = None
GRAPH_SHARD_SIZE = None
GRAPH_MANIFEST_PATH = None
if GRAPH_FORMAT == "sharded":
    GRAPH_MANIFEST_PATH = GRAPH_RESOLVED_PATH / "manifest.json"
    manifest = json.loads(GRAPH_MANIFEST_PATH.read_text(encoding="utf-8"))
    GRAPH_COUNT = int(manifest.get("num_graphs", 0) or 0)
    GRAPH_NUM_SHARDS = int(manifest.get("num_shards", 0) or 0)
    GRAPH_SHARD_SIZE = int(manifest.get("shard_size", 0) or 0)

BENCHMARK_EPOCHS = int(section_overrides["benchmark"].get("epochs", FAST_BENCHMARK_OVERRIDES["epochs"] if RUN_PROFILE == "fast" else 500))
BENCHMARK_WALK_FORWARD_MAX_FOLDS = int(section_overrides["benchmark"].get("walk_forward_max_folds", 0))
BENCHMARK_FOLD_ESTIMATE = estimate_walk_forward_folds(
    GRAPH_COUNT,
    train_frac=0.6,
    eval_frac=0.2,
    step_frac=0.1,
    min_train=128,
    min_eval=32,
    max_folds=BENCHMARK_WALK_FORWARD_MAX_FOLDS,
)
BENCHMARK_MODE_COUNT = len(BENCHMARK_MODES) if BENCHMARK_MODES else 4
BENCHMARK_WORK_UNITS = (
    BENCHMARK_EPOCHS * BENCHMARK_MODE_COUNT * BENCHMARK_FOLD_ESTIMATE
    if BENCHMARK_FOLD_ESTIMATE is not None
    else None
)
ENABLED_STAGE_NAMES = [
    name
    for enabled, name in [
        (INSTALL_DEPS, "install"),
        (RUN_BUILD_GRAPHS, "build_graphs"),
        (RUN_BENCHMARK, "benchmark"),
        (RUN_BENCHMARK, "paper_summary"),
        (RUN_TRAIN, "train"),
        (RUN_SWEEP, "sweep"),
        (RUN_DUAL_SCORE and RUN_BENCHMARK and RUN_SWEEP, "dual_score"),
        (RUN_SCENARIO and RUN_TRAIN, "scenario"),
        (RUN_SCENARIO and RUN_TRAIN, "stress"),
        (RUN_SCENARIO and RUN_TRAIN, "calibration"),
        (RUN_BACKTEST and RUN_TRAIN, "backtest"),
        (RUN_PUBLISH, "publish"),
    ]
    if enabled
]

ETA_MIN = {
    "install": 8,
    "build_graphs": 60,
    "benchmark": 45 if RUN_PROFILE == "fast" else 150,
    "paper_summary": 5,
    "train": 75 if RUN_PROFILE == "fast" else 180,
    "sweep": 40 if RUN_PROFILE == "fast" else 240,
    "dual_score": 5,
    "scenario": 75,
    "stress": 5,
    "calibration": 3,
    "backtest": 20,
    "publish": 2,
}

STAGES = []
if INSTALL_DEPS:
    STAGES.append(NotebookStage("install", "Install dependencies", eta_s=ETA_MIN["install"] * 60))
if RUN_BUILD_GRAPHS:
    STAGES.append(NotebookStage("build_graphs", "Build graphs", (str(GRAPH_OUT),), eta_s=ETA_MIN["build_graphs"] * 60))
if RUN_BENCHMARK:
    STAGES.append(NotebookStage("benchmark", "Run benchmark", (str(BENCHMARK_CSV), str(BENCHMARK_HISTORY_CSV)), eta_s=ETA_MIN["benchmark"] * 60))
    STAGES.append(NotebookStage("paper_summary", "Write paper benchmark summary", (str(PAPER_SUMMARY_MD), str(PAPER_SUMMARY_CSV), str(PAPER_SUMMARY_JSON)), eta_s=ETA_MIN["paper_summary"] * 60))
if RUN_TRAIN:
    STAGES.append(NotebookStage("train", "Train FF/BP model", (str(ENCODER_CKPT), str(CRITIC_CKPT)), eta_s=ETA_MIN["train"] * 60))
if RUN_SWEEP:
    STAGES.append(NotebookStage("sweep", "Run FF sweep", (str(SWEEP_CSV),), eta_s=ETA_MIN["sweep"] * 60))
if RUN_DUAL_SCORE and RUN_BENCHMARK and RUN_SWEEP:
    STAGES.append(NotebookStage("dual_score", "Write dual score report", (str(DUAL_SCORE_TXT), str(DUAL_SCORE_CSV)), eta_s=ETA_MIN["dual_score"] * 60))
if RUN_SCENARIO and RUN_TRAIN:
    STAGES.append(NotebookStage("scenario", "Run scenario book", (str(SCENARIO_CSV), str(SCENARIO_DIAG_CSV)), eta_s=ETA_MIN["scenario"] * 60, optional=True))
    STAGES.append(NotebookStage("stress", "Write stress report", (str(STRESS_CSV), str(STRESS_PLOT)), eta_s=ETA_MIN["stress"] * 60, optional=True))
    STAGES.append(NotebookStage("calibration", "Write hallucination calibration", (str(CALIBRATION_JSON), str(CALIBRATION_BY_TICKER_CSV)), eta_s=ETA_MIN["calibration"] * 60, optional=True))
if RUN_BACKTEST and RUN_TRAIN:
    STAGES.append(NotebookStage("backtest", "Run goodness backtest", (str(GOODNESS_CSV), str(GOODNESS_STRATEGY_CSV), str(GOODNESS_TIMELINE_PLOT)), eta_s=ETA_MIN["backtest"] * 60, optional=True))
if RUN_PUBLISH:
    STAGES.append(NotebookStage("publish", "Publish curated artifacts", eta_s=ETA_MIN["publish"] * 60, optional=True))


def show_stage_status():
    rows = stage_status_rows(MANIFEST_PATH, STAGES, root=ROOT)
    df = pd.DataFrame(rows)
    if not df.empty:
        display(df)
    print("Remaining ETA:", format_duration(remaining_eta_seconds(MANIFEST_PATH, STAGES, root=ROOT)))


print("run profile:", RUN_PROFILE)
print("run id:", RUN_ID)
print("run root:", RUN_ROOT)
print("base config:", BASE_CONFIG)
print("runtime config:", RUNTIME_CONFIG)
print("graph source base:", GRAPH_PATH_FOR_RUN)
print("graph source resolved:", GRAPH_RESOLVED_PATH, f"(format={GRAPH_FORMAT})")
print("prices path:", DATA_PRICES_PATH)
show_stage_status()


run profile: fast
run id: paper_prelim_20260413_233149
run root: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_prelim_20260413_233149
base config: /content/drive/MyDrive/Forward-Risk-Manager/configs/paper_final_500.toml
runtime config: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_prelim_20260413_233149/configs/runtime_config.toml
graph source base: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt
graph source resolved: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt.sharded (format=sharded)
prices path: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/prices.csv


,step,label,status,elapsed_s,eta_s,optional,outputs_ready
0,install,Install dependencies,pending,None,480,False,None
1,benchmark,Run benchmark,pending,None,2700,False,False
2,paper_summary,Write paper benchmark summary,pending,None,300,False,False
3,train,Train FF/BP model,pending,None,4500,False,False


Remaining ETA: 2h 13m 00s


In [3]:
preflight_rows = [
    {"item": "run_profile", "value": RUN_PROFILE},
    {"item": "enabled_stages", "value": ", ".join(ENABLED_STAGE_NAMES)},
    {"item": "base_config", "value": str(BASE_CONFIG)},
    {"item": "runtime_config", "value": str(RUNTIME_CONFIG)},
    {"item": "graph_source_base", "value": str(GRAPH_PATH_FOR_RUN)},
    {"item": "graph_source_resolved", "value": str(GRAPH_RESOLVED_PATH)},
    {"item": "graph_format", "value": GRAPH_FORMAT},
    {"item": "graph_count", "value": GRAPH_COUNT},
    {"item": "graph_num_shards", "value": GRAPH_NUM_SHARDS},
    {"item": "graph_shard_size", "value": GRAPH_SHARD_SIZE},
    {"item": "benchmark_epochs", "value": BENCHMARK_EPOCHS},
    {"item": "benchmark_mode_count", "value": BENCHMARK_MODE_COUNT},
    {"item": "benchmark_modes", "value": ", ".join(BENCHMARK_MODES) if BENCHMARK_MODES else "script defaults"},
    {"item": "benchmark_walk_forward_max_folds", "value": BENCHMARK_WALK_FORWARD_MAX_FOLDS},
    {"item": "benchmark_fold_estimate", "value": BENCHMARK_FOLD_ESTIMATE},
    {"item": "benchmark_work_units", "value": BENCHMARK_WORK_UNITS},
]
display(pd.DataFrame(preflight_rows))

if GRAPH_FORMAT == "sharded":
    print("Sharded graph artifact will be used:", GRAPH_RESOLVED_PATH)
    if GRAPH_MANIFEST_PATH is not None:
        print("Sharded manifest:", GRAPH_MANIFEST_PATH)
elif SHARDED_GRAPH_PATH.is_dir():
    print("Sharded sidecar exists and will be preferred by the training/benchmark scripts:", SHARDED_GRAPH_PATH)
else:
    print("Using packed graph artifact:", GRAPH_RESOLVED_PATH)
    print(
        "Recommendation: run notebooks/shard_graph_artifact_colab.ipynb before large runs "
        "to reduce load latency and memory pressure."
    )

if RUN_PROFILE == "fast":
    print("Fast profile keeps sweep/scenario/backtest out of the default rerun path.")
else:
    print("Full profile keeps the broader paper pipeline enabled.")

if BENCHMARK_WORK_UNITS is not None:
    print(
        "Rough benchmark work estimate:",
        f"{BENCHMARK_EPOCHS} epochs x {BENCHMARK_MODE_COUNT} modes x {BENCHMARK_FOLD_ESTIMATE} folds = {BENCHMARK_WORK_UNITS}",
    )
else:
    print("Rough benchmark work estimate unavailable because graph count could not be inferred.")


,item,value
0,run_profile,fast
1,enabled_stages,"install, benchmark, paper_summary, train"
2,base_config,/content/drive/MyDrive/Forward-Risk-Manager/co...
3,runtime_config,/content/drive/MyDrive/Forward-Risk-Manager/ru...
4,graph_source_base,/content/drive/MyDrive/Forward-Risk-Manager/da...
5,graph_source_resolved,/content/drive/MyDrive/Forward-Risk-Manager/da...
6,graph_format,sharded
7,graph_count,10400
8,graph_num_shards,41
9,graph_shard_size,256


Sharded graph artifact will be used: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt.sharded
Sharded manifest: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt.sharded/manifest.json
Fast profile keeps sweep/scenario/backtest out of the default rerun path.
Rough benchmark work estimate: 15 epochs x 3 modes x 2 folds = 90


In [4]:
required_modules = ["torch", "torch_geometric", "pandas", "numpy", "tqdm", "matplotlib"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if INSTALL_DEPS or missing:
    install_cmd = (
        f"{PYTHON_EXE} -m pip install --upgrade pip setuptools wheel && "
        f"{PYTHON_EXE} -m pip install -r requirements.txt && "
        f"{PYTHON_EXE} -m pip install -e ."
    )
    run_tracked_command(
        step="install",
        label="Install dependencies",
        command=install_cmd,
        manifest_path=MANIFEST_PATH,
        eta_s=ETA_MIN["install"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        metadata={"missing_modules": missing},
    )
else:
    print("Dependencies already installed. Skipping install stage.")

show_stage_status()



/usr/bin/python3 -m pip install --upgrade pip setuptools wheel && /usr/bin/python3 -m pip install -r requirements.txt && /usr/bin/python3 -m pip install -e .
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.1 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.7 MB/s  0:00:00
Obtaining file:///content/drive/MyDrive/Forward-Risk-Manager
  I

,step,label,status,elapsed_s,eta_s,optional,outputs_ready
0,install,Install dependencies,completed,20.475842,480.0,False,None
1,benchmark,Run benchmark,pending,NaN,2700.0,False,False
2,paper_summary,Write paper benchmark summary,pending,NaN,300.0,False,False
3,train,Train FF/BP model,pending,NaN,4500.0,False,False


Remaining ETA: 2h 05m 00s


In [5]:
if RUN_BUILD_GRAPHS:
    run_tracked_command(
        step="build_graphs",
        label="Build graphs",
        command=f"{PYTHON_EXE} scripts/build_graphs.py --config {shell_quote(str(RUNTIME_CONFIG))}",
        manifest_path=MANIFEST_PATH,
        required_outputs=[GRAPH_OUT],
        eta_s=ETA_MIN["build_graphs"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )
else:
    if not GRAPH_RESOLVED_PATH.exists():
        raise FileNotFoundError(f"Missing graph artifact: {GRAPH_RESOLVED_PATH}")
    print("Using existing graph artifact:", GRAPH_RESOLVED_PATH, f"(format={GRAPH_FORMAT})")

if RUN_BENCHMARK:
    benchmark_command = f"{PYTHON_EXE} scripts/benchmark_training.py --config {shell_quote(str(RUNTIME_CONFIG))}"
    if BENCHMARK_MODES:
        benchmark_command += f" --modes {shell_quote(','.join(BENCHMARK_MODES))}"
    run_tracked_command(
        step="benchmark",
        label="Run benchmark",
        command=benchmark_command,
        manifest_path=MANIFEST_PATH,
        required_outputs=[BENCHMARK_CSV, BENCHMARK_HISTORY_CSV, BENCHMARK_PLOT, BENCHMARK_BAR],
        eta_s=ETA_MIN["benchmark"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )
    run_tracked_command(
        step="paper_summary",
        label="Write paper benchmark summary",
        command=(
            f"{PYTHON_EXE} scripts/paper_benchmark_summary.py "
            f"--benchmark {shell_quote(str(BENCHMARK_CSV))} "
            f"--folds-csv {shell_quote(str(BENCHMARK_FOLDS_CSV))} "
            f"--out-md {shell_quote(str(PAPER_SUMMARY_MD))} "
            f"--out-csv {shell_quote(str(PAPER_SUMMARY_CSV))} "
            f"--out-json {shell_quote(str(PAPER_SUMMARY_JSON))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[PAPER_SUMMARY_MD, PAPER_SUMMARY_CSV, PAPER_SUMMARY_JSON],
        eta_s=ETA_MIN["paper_summary"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )

if RUN_TRAIN:
    run_tracked_command(
        step="train",
        label="Train FF/BP model",
        command=f"{PYTHON_EXE} scripts/train_ff_gnn.py --config {shell_quote(str(RUNTIME_CONFIG))}",
        manifest_path=MANIFEST_PATH,
        required_outputs=[MODEL_CKPT, ENCODER_CKPT, CRITIC_CKPT, TRAIN_LOG_CSV],
        eta_s=ETA_MIN["train"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )

if RUN_SWEEP:
    run_tracked_command(
        step="sweep",
        label="Run FF sweep",
        command=f"{PYTHON_EXE} scripts/ff_sweep.py --config {shell_quote(str(RUNTIME_CONFIG))}",
        manifest_path=MANIFEST_PATH,
        required_outputs=[SWEEP_CSV],
        eta_s=ETA_MIN["sweep"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )

if RUN_DUAL_SCORE and RUN_BENCHMARK and RUN_SWEEP:
    run_tracked_command(
        step="dual_score",
        label="Write dual score report",
        command=(
            f"{PYTHON_EXE} scripts/dual_score_report.py "
            f"--benchmark {shell_quote(str(BENCHMARK_CSV))} "
            f"--sweep {shell_quote(str(SWEEP_CSV))} "
            f"--out {shell_quote(str(DUAL_SCORE_TXT))} "
            f"--out-csv {shell_quote(str(DUAL_SCORE_CSV))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[DUAL_SCORE_TXT, DUAL_SCORE_CSV],
        eta_s=ETA_MIN["dual_score"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )

if RUN_SCENARIO and RUN_TRAIN:
    run_tracked_command(
        step="scenario",
        label="Run scenario book",
        command=(
            f"{PYTHON_EXE} scripts/scenario_book.py "
            f"--config {shell_quote(str(RUNTIME_CONFIG))} "
            f"--critic-model {shell_quote(str(CRITIC_CKPT))} "
            f"--out {shell_quote(str(SCENARIO_CSV))} "
            f"--diag-out {shell_quote(str(SCENARIO_DIAG_CSV))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[SCENARIO_CSV, SCENARIO_DIAG_CSV],
        eta_s=ETA_MIN["scenario"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )
    run_tracked_command(
        step="stress",
        label="Write stress report",
        command=(
            f"{PYTHON_EXE} scripts/stress_test_report.py "
            f"--csv {shell_quote(str(SCENARIO_CSV))} "
            f"--out-csv {shell_quote(str(STRESS_CSV))} "
            f"--out-plot {shell_quote(str(STRESS_PLOT))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[STRESS_CSV, STRESS_PLOT],
        eta_s=ETA_MIN["stress"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )
    run_tracked_command(
        step="calibration",
        label="Write hallucination calibration",
        command=(
            f"{PYTHON_EXE} scripts/hallucination_calibration.py "
            f"--csv {shell_quote(str(SCENARIO_CSV))} "
            f"--out {shell_quote(str(CALIBRATION_JSON))} "
            f"--out-by-ticker {shell_quote(str(CALIBRATION_BY_TICKER_CSV))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[CALIBRATION_JSON, CALIBRATION_BY_TICKER_CSV],
        eta_s=ETA_MIN["calibration"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )

if RUN_BACKTEST and RUN_TRAIN:
    run_tracked_command(
        step="backtest",
        label="Run goodness backtest",
        command=(
            f"{PYTHON_EXE} scripts/goodness_backtest.py "
            f"--config {shell_quote(str(RUNTIME_CONFIG))} "
            f"--prices {shell_quote(str(DATA_PRICES_PATH))} "
            f"--out-csv {shell_quote(str(GOODNESS_CSV))} "
            f"--out-quantiles {shell_quote(str(GOODNESS_QUANTILES_CSV))} "
            f"--out-plot {shell_quote(str(GOODNESS_PLOT))} "
            f"--out-events {shell_quote(str(GOODNESS_EVENTS_CSV))} "
            f"--out-strategy {shell_quote(str(GOODNESS_STRATEGY_CSV))} "
            f"--out-timeline {shell_quote(str(GOODNESS_TIMELINE_PLOT))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[GOODNESS_CSV, GOODNESS_STRATEGY_CSV, GOODNESS_TIMELINE_PLOT],
        eta_s=ETA_MIN["backtest"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )

if RUN_PUBLISH:
    run_tracked_command(
        step="publish",
        label="Publish curated artifacts",
        command=f"{PYTHON_EXE} scripts/publish_run.py --run-id {shell_quote(RUN_ID)}",
        manifest_path=MANIFEST_PATH,
        eta_s=ETA_MIN["publish"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )

show_stage_status()


Using existing graph artifact: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt.sharded (format=sharded)

/usr/bin/python3 scripts/benchmark_training.py --config /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_prelim_20260413_233149/configs/runtime_config.toml --modes ff_layerwise,ff_e2e,backprop_contrastive
graph artifact: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt.sharded (format=sharded)
econ ticker: requested=SPY,MDY,BWC,AAON,AUTO effective=SPY source=requested_priority rows=5282
portfolio ticker: requested=AUTO effective=IBM source=auto_max_rows rows=16140 horizon=21
walk-forward splits=2 (train_frac=0.6, eval_frac=0.2, step_frac=0.1)
benchmark resiliency: retry_safe_on_error=True, continue_on_mode_error=True, torch_compile=False (mode=max-autotune-no-cudagraphs)
benchmark seeds: [7]

Benchmark: 100%|██████████| 15/15 [05:58<00:00, 23.89s/epoch]
calibrated goodness_target=10.6277 (train-

: 

In [ ]:
ARTIFACTS = [
    RUNTIME_CONFIG,
    GRAPH_PATH_FOR_RUN,
    TRAIN_LOG_CSV,
    TRAIN_PLOT,
    MODEL_CKPT,
    ENCODER_CKPT,
    CRITIC_CKPT,
    BENCHMARK_CSV,
    BENCHMARK_FOLDS_CSV,
    BENCHMARK_BASELINE_CSV,
    BENCHMARK_HISTORY_CSV,
    BENCHMARK_PLOT,
    BENCHMARK_BAR,
    PAPER_SUMMARY_MD,
    PAPER_SUMMARY_CSV,
    PAPER_SUMMARY_JSON,
    SWEEP_CSV,
    DUAL_SCORE_TXT,
    DUAL_SCORE_CSV,
    SCENARIO_CSV,
    SCENARIO_DIAG_CSV,
    STRESS_CSV,
    STRESS_PLOT,
    CALIBRATION_JSON,
    CALIBRATION_BY_TICKER_CSV,
    GOODNESS_CSV,
    GOODNESS_QUANTILES_CSV,
    GOODNESS_PLOT,
    GOODNESS_EVENTS_CSV,
    GOODNESS_STRATEGY_CSV,
    GOODNESS_TIMELINE_PLOT,
    MANIFEST_PATH,
]

rows = []
for path in ARTIFACTS:
    rows.append({
        "path": str(path),
        "exists": path.exists(),
        "bytes": path.stat().st_size if path.exists() and path.is_file() else None,
    })

display(pd.DataFrame(rows))
show_stage_status()

if PAPER_SUMMARY_MD.exists():
    print("\nPaper summary markdown path:", PAPER_SUMMARY_MD)
if MANIFEST_PATH.exists():
    print("Stage manifest:", MANIFEST_PATH)
    print(MANIFEST_PATH.read_text()[:4000])
